# Normalizer forward and inverse roundtrip

This notebook confirms that `Normalizer.normalize_input` followed by `denormalize_input`
recovers the original tensor, including the log1p and expm1 branch, and that the forward
transform centres the data as intended. The same is checked for the output (parameter) path.

Modules exercised:

- `pipelines.dataset_pipeline.normalization.Normalizer`
- `pipelines.dataset_pipeline.normalization.StatsComputer`
- `pipelines.dataset_pipeline.normalization.Stats`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Build a dataset and fit input statistics

We reuse the standard synthetic stack and fit input statistics, then build a `Normalizer`.


In [ ]:
from tools.monitoring.logger                             import Logger
from configuration.dataset_config             import InputConfig, OutputConfig
from tools.data.representation             import Representation
from pipelines.dataset_pipeline.spatial       import Patcher
from pipelines.dataset_pipeline.datasets      import PatchDataset
from pipelines.dataset_pipeline.normalization import StatsComputer, Normalizer, Stats

logger           = Logger(log_dir="/tmp/dataset_nb_logs", name="nb08", level="WARNING")
n_secondaries    = 1
n_interferograms = 2
n_passes         = 1 + n_secondaries + n_interferograms
H, W             = 40, 40

rng   = np.random.default_rng(SEED)
stack = (2.0 * rng.random((n_passes, H, W)) * np.exp(1j * rng.uniform(-np.pi, np.pi, (n_passes, H, W)))).astype(np.complex64)
gt    = np.abs(rng.standard_normal((3, H, W))).astype(np.float32)

input_config = InputConfig(use_primary=True, primary_representation=Representation.MAG_ANGLE,
                           use_secondaries=True, secondaries_representation=Representation.MAG_ONLY,
                           use_interferograms=True, interferograms_representation=Representation.ANGLE_ONLY,
                           use_dem=False)
patcher = Patcher.build(spatial_size=(H, W), patch_size=(H, W), stride=H)
dataset = PatchDataset(inputs=stack, gt_parameters=gt, grid=patcher, input_config=input_config,
                       output_config=OutputConfig(), split_name="val",
                       n_secondaries=n_secondaries, n_interferograms=n_interferograms, n_gaussians=1)

stats      = StatsComputer.compute_input_stats(dataset=dataset, logger=logger, input_config=input_config,
                                               n_secondaries=n_secondaries, n_interferograms=n_interferograms,
                                               num_workers=0, batch_size=8)
normalizer = Normalizer(stats)



## Forward transform and inverse transform

We pull the raw (un-normalized) tensor by temporarily detaching the normalizer, then normalize
and denormalize it.


In [ ]:
raw_tensor, _ = dataset[0]
normalized    = normalizer.normalize_input(raw_tensor)
recovered     = normalizer.denormalize_input(normalized)

roundtrip_err = float(np.abs(recovered - raw_tensor).max())
print("normalized mean per channel:", np.round(normalized.reshape(normalized.shape[0], -1).mean(axis=1), 3))
print("roundtrip max abs error    :", f"{roundtrip_err:.3e}")


In [ ]:
ch = 0
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, data, title in zip(axes, [raw_tensor[ch], normalized[ch], recovered[ch]],
                           ["raw", "normalized", "denormalized"]):
    im = ax.imshow(data, cmap="viridis")
    ax.set_title(f"{title} (ch {ch})")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()



## Scatter of recovered versus original

A perfect inverse lies on the identity line for every channel.


In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.scatter(raw_tensor.ravel(), recovered.ravel(), s=2, alpha=0.3)
lo = min(raw_tensor.min(), recovered.min()); hi = max(raw_tensor.max(), recovered.max())
ax.plot([lo, hi], [lo, hi], "C3--", label="identity")
ax.set_xlabel("original value"); ax.set_ylabel("recovered value")
ax.set_title("denormalize(normalize(x)) vs x")
ax.legend()
plt.tight_layout()
plt.show()



## Output path roundtrip

We fit output statistics from a parameter file and confirm the parameter normalization is also
invertible.


In [ ]:
import tempfile, os

n_gaussians = 1
params      = np.abs(rng.standard_normal((n_gaussians * 3, H, W))).astype(np.float32)
params_path = tempfile.mktemp(suffix=".npy")
np.save(params_path, params)

out_stats   = StatsComputer.compute_output_stats(params_path=params_path, n_gaussians=n_gaussians,
                                                 output_config=OutputConfig(), logger=None)
out_norm    = Normalizer(Stats(input_stats=None, output_stats=out_stats.output_stats))

selected    = OutputConfig().selected_indices(n_gaussians)
gt_sel      = params[selected].astype(np.float32)
gt_round    = out_norm.denormalize_output(out_norm.normalize_output(gt_sel))
print("output roundtrip max abs error:", f"{float(np.abs(gt_round - gt_sel).max()):.3e}")
os.remove(params_path)


## Expected visual outcome

The denormalized panel should be visually identical to the raw panel, and the roundtrip error
for both the input and output paths should be near machine precision. The scatter plot should
collapse onto the dashed identity line. Normalized channel means should sit near the strategy
target (close to zero for z-score-like channels).